# CLIP — Contrastive Language–Image Pre-training

A self-contained refresher. CLIP (OpenAI, 2021) learns a *shared* embedding space for
images and text so you can compare a picture and a sentence with a dot product.

**Domain:** Architectures · **recommended addition** · **runnable:** yes

## 1. What & Why

**What.** CLIP is a pair of encoders — one for images (a ResNet or ViT), one for text (a
Transformer) — trained together so that an image and its caption land at nearly the same
point in a shared vector space. It was trained contrastively on ~400M noisy
(image, alt-text) pairs scraped from the web, with **no** hand-labeled class taxonomy.

**The problem it solves.** Classic image classifiers are locked to a fixed label set
(ImageNet's 1000 classes). Want a new class? Collect labels and retrain. CLIP turns
classification into a *similarity* problem: embed the image once, embed any text prompts you
like (`"a photo of a cat"`, `"a satellite image of a forest"`), and pick the closest. This
gives **zero-shot** classification over arbitrary, open-vocabulary labels with no task-specific
training.

**Reach for CLIP when** you need: zero-shot / open-vocabulary image classification,
image↔text retrieval (search photos by caption or vice-versa), a quick semantic similarity
score between an image and text, or a frozen vision backbone for a larger multimodal system
(it powers Stable Diffusion's text conditioning, LLaVA-style VLMs, and more).

**Skip CLIP when** the task is pixel-level (segmentation, depth), needs precise counting /
spatial reasoning / OCR, or you already have abundant labeled data for a closed set — a
fine-tuned supervised ViT will usually beat zero-shot CLIP on its home turf.

## 2. Mental Model

Picture **one shared hypersphere**. The image encoder maps a picture to a point on it; the
text encoder maps a sentence to a point on it. Training is a tug-of-war: pull a matched
(image, caption) pair together, push every mismatched pair apart. After training, "closeness"
(cosine similarity) means "these describe the same thing" — regardless of whether the input
was pixels or words. CLIP is essentially a **learned bilingual dictionary** between the
language of images and the language of text.

The training signal lives in a single **N×N similarity matrix** for a batch of N pairs:

```
            text_0  text_1  text_2  ...  text_N
  image_0  [  ✓                              ]   <- maximize the diagonal
  image_1  [          ✓                      ]   <- everything off-diagonal is a negative
  image_2  [                  ✓              ]
   ...
```

Each row should softmax-peak on its diagonal entry (the right caption), and so should each
column (the right image). That symmetric cross-entropy *is* the contrastive (InfoNCE) loss.
Crucially, the other items in the batch are the negatives — a big batch = many negatives =
a sharper space, which is why CLIP was trained with batch sizes in the tens of thousands.

## 3. Key Concepts

- **Dual encoders.** Independent image and text towers. They never attend to each other;
  they only meet at the final dot product. This "late fusion" is what makes retrieval cheap —
  you can pre-compute and index embeddings.
- **L2-normalized embeddings.** Both outputs are projected to a common dim and normalized to
  unit length, so the dot product *is* cosine similarity, bounded in [-1, 1].
- **Contrastive / InfoNCE loss.** Symmetric cross-entropy over the similarity matrix:
  image→text loss + text→image loss, averaged. Labels are just the diagonal indices.
- **Temperature (logit scale).** Similarities are divided by a learned temperature `τ`
  (stored as `logit_scale`, applied as `exp(logit_scale)`) before softmax. Low τ = sharp,
  confident distribution; it's clamped during training to avoid blowing up.
- **Zero-shot classification.** Build a text prompt per class, embed them once into a
  "classifier matrix," then for each image take softmax over `image · class_texts`.
- **Prompt engineering & ensembling.** `"a photo of a {label}"` beats the bare label;
  averaging the embeddings of many prompt templates ("a blurry photo of a {label}", "a sculpture
  of a {label}", …) reliably bumps accuracy a point or two.
- **Variants.** Model names like `ViT-B/32` = ViT-Base, 32px patches. **OpenCLIP** reproduces
  and scales CLIP on open data (LAION). **SigLIP** swaps the softmax for a sigmoid loss, which
  trains well at smaller batch sizes and often beats CLIP at matched scale.

## 4. Setup

The worked examples below are **pure NumPy** — they run anywhere with no model download, no GPU,
no network. They reimplement CLIP's *math* (contrastive loss, zero-shot scoring) on toy vectors
so the mechanics are visible.

```bash
pip install numpy
```

To run a **real** pretrained CLIP, install one of these (each pulls in PyTorch + downloads
weights, so the example cell is gated behind an env var and skipped by default):

```bash
pip install open_clip_torch pillow          # OpenCLIP — recommended, many checkpoints
# or
pip install transformers torch pillow       # Hugging Face CLIPModel
```

In [1]:
import numpy as np

rng = np.random.default_rng(0)
print("numpy", np.__version__)


def l2_normalize(x, axis=-1, eps=1e-8):
    """Project rows onto the unit hypersphere so dot product == cosine similarity."""
    return x / (np.linalg.norm(x, axis=axis, keepdims=True) + eps)


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

numpy 2.4.6


## 5. Worked Examples

### Example 1 — the contrastive loss, by hand

We fake the *output* of the two encoders: `D`-dim embeddings for a batch of `N` images and
their `N` matching captions. CLIP normalizes both, builds the similarity matrix scaled by a
temperature, and applies symmetric cross-entropy where the correct answer for row *i* is
column *i* (the diagonal). To make the point, we hand the matched pairs a shared "signal" so
the diagonal genuinely stands out.

In [2]:
N, D = 6, 32                       # batch of 6 pairs, 32-dim embedding space
temperature = 0.07                # CLIP's learned temperature lives around here

# Each pair shares a latent "concept" vector; encoders add their own noise on top.
concepts   = rng.standard_normal((N, D))
img_emb  = l2_normalize(concepts + 0.35 * rng.standard_normal((N, D)))
txt_emb  = l2_normalize(concepts + 0.35 * rng.standard_normal((N, D)))

# Similarity matrix (cosine, since rows are unit-norm), scaled by 1/temperature.
logits = (img_emb @ txt_emb.T) / temperature      # shape (N, N)

# Symmetric InfoNCE: each image should pick its own caption, and vice-versa.
labels = np.arange(N)
p_i2t  = softmax(logits, axis=1)                  # image -> text distribution
p_t2i  = softmax(logits, axis=0)                  # text  -> image distribution
loss_i2t = -np.log(p_i2t[labels, labels]).mean()
loss_t2i = -np.log(p_t2i[labels, labels]).mean()
loss = 0.5 * (loss_i2t + loss_t2i)

print(f"diagonal (matched) avg cosine : {(img_emb * txt_emb).sum(1).mean():+.3f}")
print(f"off-diagonal     avg cosine   : {logits[~np.eye(N, dtype=bool)].mean()*temperature:+.3f}")
print(f"image->text accuracy (argmax) : {(p_i2t.argmax(1) == labels).mean():.0%}")
print(f"symmetric contrastive loss    : {loss:.4f}")

diagonal (matched) avg cosine : +0.902
off-diagonal     avg cosine   : -0.027
image->text accuracy (argmax) : 100%
symmetric contrastive loss    : 0.0001


Matched pairs sit far closer than random pairs, so every image's argmax lands on its own
caption and the loss is small. Two things to feel here: (1) the loss is just cross-entropy
with the **diagonal as the label**, and (2) the other items in the batch *are* the negatives —
a bigger batch means more (and harder) negatives, which is why CLIP wants huge batches.

### Example 2 — zero-shot classification, the CLIP way

No training. We embed one text prompt per candidate class, embed an image, and softmax over
`image · class_prompts`. Below we simulate a 3-way "pet" classifier: the image embedding is
built to lean toward the *cat* concept, and CLIP-style scoring recovers it. Note the
`"a photo of a {label}"` prompt template — CLIP was trained on captions, so a full sentence
matches the training distribution far better than a bare word.

In [3]:
labels_text = ["cat", "dog", "car"]
prompts = [f"a photo of a {w}" for w in labels_text]

# Stand-in text encoder: each prompt -> a fixed concept direction.
concept_dirs = l2_normalize(rng.standard_normal((len(prompts), D)))
text_features = concept_dirs                      # (n_classes, D), already unit-norm

# Stand-in image: mostly the "cat" direction (index 0) plus a little noise.
image = l2_normalize(0.8 * concept_dirs[0] + 0.2 * rng.standard_normal(D))

# CLIP inference: cosine sims -> temperature scale -> softmax = class probabilities.
sims = text_features @ image                       # (n_classes,)
probs = softmax(sims / temperature)

for label, p in sorted(zip(labels_text, probs), key=lambda t: -t[1]):
    print(f"  {label:>3} : {p:6.1%}")
print("\nprediction:", labels_text[int(probs.argmax())])

  cat :  52.0%
  dog :  47.2%
  car :   0.9%

prediction: cat


Swap `labels_text` for any vocabulary you want — that open-endedness is the whole point.
Two caveats the math makes obvious: the softmax is only over the labels **you** supply, so
probabilities are *relative confidences within your candidate set*, not calibrated
"is this even a pet?" scores; and the result is only as good as your prompts (hence prompt
engineering and ensembling).

### Example 3 — a real pretrained CLIP (gated, optional)

The cells above show the mechanics; a real model just gives you *good* encoders. This cell is
skipped unless you set `RUN_CLIP=1` and have `open_clip_torch` installed, so the notebook still
executes top-to-bottom offline. It shows the canonical call shape for genuine zero-shot
classification.

In [4]:
import os, importlib.util

if os.getenv("RUN_CLIP") and importlib.util.find_spec("open_clip") is not None:
    import open_clip
    import torch
    from PIL import Image

    model, _, preprocess = open_clip.create_model_and_transforms(
        "ViT-B-32", pretrained="laion2b_s34b_b79k")
    tokenizer = open_clip.get_tokenizer("ViT-B-32")
    model.eval()

    classes = ["a photo of a cat", "a photo of a dog", "a photo of a car"]
    image = preprocess(Image.open("example.jpg")).unsqueeze(0)
    text = tokenizer(classes)

    with torch.no_grad():
        img_f = model.encode_image(image)
        txt_f = model.encode_text(text)
        img_f /= img_f.norm(dim=-1, keepdim=True)
        txt_f /= txt_f.norm(dim=-1, keepdim=True)
        probs = (100.0 * img_f @ txt_f.T).softmax(dim=-1)   # 100 ~ exp(logit_scale)
    print("class probs:", probs.tolist())
else:
    print("Skipped: set RUN_CLIP=1 and `pip install open_clip_torch pillow` to run the real model.")
    print("Call shape above is exactly how OpenCLIP does zero-shot classification.")

Skipped: set RUN_CLIP=1 and `pip install open_clip_torch pillow` to run the real model.
Call shape above is exactly how OpenCLIP does zero-shot classification.


## 6. Gotchas & Pitfalls

- **Forgetting to L2-normalize.** Without unit-norm embeddings the dot product is *not* cosine
  similarity and the temperature scale is meaningless. Both towers must be normalized before
  the matmul.
- **Ignoring the temperature / logit scale.** Real CLIP applies `exp(logit_scale)` (≈100) to
  the cosine similarities before softmax. Skip it and your probabilities are mush; the example
  uses `1/temperature` to the same effect.
- **Bare-word prompts.** `"cat"` underperforms `"a photo of a cat"`. CLIP saw captions, not
  class names. For a real accuracy bump, **ensemble** many templates and average their
  (normalized) text embeddings into one class vector.
- **Treating softmax probs as calibrated.** The softmax is over *your* candidate prompts only.
  Add an unrelated class and every probability shifts. CLIP gives you a ranking, not a
  "none of the above" detector — for that, threshold the raw cosine similarity.
- **77-token text limit.** CLIP's text encoder truncates at 77 tokens. Long descriptions get
  silently cut; keep prompts short.
- **Preprocessing mismatch.** Use the exact `preprocess` transform that ships with the
  checkpoint (resize, center-crop, the specific mean/std). Feeding raw or differently-normalized
  pixels quietly tanks accuracy.
- **Weak at counting, spatial relations, OCR, fine-grained categories.** "Three dogs to the
  left of a car" is not CLIP's strength; it captures *bag-of-concepts* gist more than precise
  structure. Don't expect reliable counting or text reading.
- **Batch size matters in training.** Negatives come from the batch, so small-batch contrastive
  training underperforms. If you must train small, prefer SigLIP's sigmoid loss.

## 7. When to Use vs Alternatives

| Approach | Best for | Trade-off vs CLIP |
|---|---|---|
| **CLIP (zero-shot)** | open-vocab classification, image↔text retrieval, text-conditioning backbone | no training needed; loses to a fine-tuned model when you have ample labels |
| **Supervised ViT/ResNet** | closed label set with plenty of labels | higher accuracy on that set; cannot generalize to unseen classes without retraining |
| **OpenCLIP** | same API, open (LAION) data, many sizes incl. very large | reproducible & often stronger; just a CLIP you can pick checkpoints for |
| **SigLIP** | strong embeddings, trained at modest batch sizes | sigmoid loss → better small-batch behavior, usually higher zero-shot at matched scale |
| **DINOv2 (self-supervised vision)** | pure-vision features, segmentation, dense tasks | no text tower → no zero-shot-by-prompt, but richer spatial features |
| **BLIP / captioning VLMs** | *generating* text about an image (captions, VQA) | generative, not a fast similarity index; heavier per query |
| **Fine-tuned / LoRA CLIP** | domain shift (medical, satellite) where zero-shot is weak | recovers accuracy on niche domains at the cost of some generality |

Rule of thumb: **retrieval, open-vocabulary, or "I have no labels yet" → CLIP/OpenCLIP/SigLIP.**
**Fixed classes with a labeled dataset → fine-tune a supervised backbone.** **Need sentences
out, not similarity scores → a generative VLM.**

## 8. Resources

- **Paper — *Learning Transferable Visual Models From Natural Language Supervision* (Radford et al., 2021):** https://arxiv.org/abs/2103.00020
- **OpenAI CLIP repo (original code + weights):** https://github.com/openai/CLIP
- **OpenCLIP (open reproduction, many checkpoints):** https://github.com/mlfoundations/open_clip
- **Hugging Face CLIP docs (`CLIPModel`, processors):** https://huggingface.co/docs/transformers/model_doc/clip
- **SigLIP — *Sigmoid Loss for Language Image Pre-Training* (Zhai et al., 2023):** https://arxiv.org/abs/2303.15343
- **OpenAI blog post (intuition + zero-shot results):** https://openai.com/research/clip